In [1]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [2]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

Available VISA resources:
 - USB0::0x0957::0x1F01::MY53270560::0::INSTR
 - USB0::0x0957::0x1F01::MY57280340::0::INSTR
 - USB0::0x2A8D::0x1202::MY59003914::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391394::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391395::0::INSTR
Connected to: Keysight Technologies,E36313A,MY59003914,2.1.0-1.0.4-1.12


In [3]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

[Client] Connected to 192.168.1.10:7
Default Signals Set
[128]


In [4]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

['+2.99835900E+00', '+5.49300000E-03']
Default Signals Set
[128]


In [5]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

['+2.99818600E+00', '+1.27340000E-02']


In [6]:
#Turn on CLK 
Device_ID_CLK = "USB0::0x0957::0x1F01::MY57280340::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

Connected to: Agilent Technologies, N5173B, MY57280340, B.01.65


In [7]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_LM["id"]
scan_len = Utils.DL_EN_SC_LM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
scan_data[1] = 0
scan_data[17] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [8]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_RM["id"]
scan_len = Utils.DL_EN_SC_RM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
scan_data[19] = 0
scan_data[61] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [9]:
#Control Logic TD Data
import numpy as np
import random
import re
def scan_gen(time):
    SC_data = f"""
    IN_EXT_LOC = 1'b0;
    TDC_EN_EXT_LOC = 1'b0;
    ENCHG_EXT_LOC = 1'b1;
    ENTIME_EXT_LOC = 1'b1;
    PCHG_EXT_LOC = 1'b0;
    VDAC_CTRL_EXT_LOC = 1'b1;
    DEL_RST_EXT_LOC = 1'b0;
    TDC_COMPUTE_EXT_LOC = 1'b0;
    TDC_RST_EXT_LOC = 1'b0;
    VTC_EN_EXT_LOC = 1'b1;

    IN_LB = 6'd3;
    IN_UB = 6'd22; 
    TDC_EN_LB = 6'd1;
    TDC_EN_UB = 6'd21;
    ENCHG_LB = 6'd0;
    ENCHG_UB = 6'd0;
    ENTIME_LB = 6'd0;
    ENTIME_UB = 6'd0;
    PCHG_LB = 6'd2;
    PCHG_UB = 6'd22;
    VDAC_CTRL_LB = 6'd0;
    VDAC_CTRL_UB = 6'd0; 
    DEL_RST_LB = 6'd2;
    DEL_RST_UB = 6'd22;
    TDC_COMPUTE_LB = 6'd19;
    TDC_COMPUTE_UB = 6'd24;
    TDC_RST_LB = 6'd1;
    TDC_RST_UB = 6'd21;
    BL_NUM_TGT = 2'd3;
    BL_DONE_TIME = 6'd55;
    VTC_EN_LB = 6'd0;
    VTC_EN_UB = 6'd0;
    SAMPLE_EDGE_TIME = 6'd{time};

    CHG_TIME = 1'd0;
    OSC_DEL = 1'd0;
    """
    SC_order = ["BL_DONE_TIME","SAMPLE_EDGE_TIME", "BL_NUM_TGT", "BLANK_4", "IN_LB", "TDC_EN_LB", "ENCHG_LB", "IN_EXT_LOC","TDC_EN_EXT_LOC","ENCHG_EXT_LOC",\
                "ENTIME_EXT_LOC","PCHG_EXT_LOC","VDAC_CTRL_EXT_LOC", 'ENTIME_LB', 'PCHG_LB', 'VDAC_CTRL_LB', 'DEL_RST_LB', 'TDC_COMPUTE_LB', 'TDC_RST_LB', \
                'VTC_EN_LB',"DEL_RST_EXT_LOC","TDC_COMPUTE_EXT_LOC","TDC_RST_EXT_LOC","VTC_EN_EXT_LOC","CHG_TIME","OSC_DEL", "VTC_EN_UB","TDC_RST_UB","TDC_COMPUTE_UB",\
                "DEL_RST_UB","VDAC_CTRL_UB","PCHG_UB","ENTIME_UB","ENCHG_UB","TDC_EN_UB","IN_UB","BLANK_12_PISO"]

    lines = SC_data.splitlines()
    sc_signal = {}
    for line in lines:
        if line.strip():  # Skip empty lines
            # Handle 'd' (decimal) format
            match = re.match(r"(\w+) = (\d+)'d([0-9]+);", line.strip())
            if match:
                signal_name = match.group(1)
                num_bits = int(match.group(2))  # Number of bits
                signal_value = match.group(3)  # Decimal value


                # If there's only 1 bit, do not add the bit-range
                if num_bits == 1:
                    name = signal_name
                else:
                    # Format the name with the bit-range (e.g., VTC_EN_UB<[5:0]>)
                    name = "%s" % (signal_name) # Using 0-based index for the range

                # Format the decimal value to binary and pad to the correct number of bits
                binary_value = bin(int(signal_value))[2:].zfill(num_bits)
                sc_signal[name] = binary_value


            # Handle 'b' (binary) format
            elif "b" in line:  # e.g., BL_NUM_TGT = 2'b01;
                match = re.match(r"(\w+) = (\d+)'b([01]+);", line.strip())
                if match:
                    signal_name = match.group(1)
                    num_bits = int(match.group(2))  # Number of bits
                    signal_value = match.group(3)  # Binary value

                    # If there's only 1 bit, do not add the bit-range
                    if num_bits == 1:
                        name = signal_name
                    else:
                        # Format the name with the bit-range (e.g., BL_NUM_TGT<[1:0]> for 2 bits)
                        name = "%s" % (signal_name)

                    # Format the binary value to match the number of bits
                    binary_value = signal_value.zfill(num_bits)
                    sc_signal[name] = binary_value


    # print(sc_signal)
    output_list = []
    for signal in SC_order:
        if "BLANK" in signal:
            num_blanks = re.findall(r'\d+', signal)
            output_list += [0]*int(num_blanks[0])
        else:
            temp = sc_signal[signal][::-1]
            output_list += list(map(int,temp))
    print(sc_signal["SAMPLE_EDGE_TIME"])
    output_list = output_list[::-1] + [0]*30
    return output_list

def set_CS(cs):
    ret_data = 0
    if cs == 0:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 1:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 2:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 3:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    return ret_data

In [10]:
#Call Scan IN here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_data = scan_gen(5)
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
#print(ret_data)

000101
[162, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received


In [11]:
#Initialise SRAM with default data.
write_data = Utils.get_Default_Write_Data().tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [ ]:
'''
#SRAM IMC DATA
arr = np.zeros((320, 1024),dtype=np.uint8)
#Write all reference lines at 128
for refline in range(24,280,36):
    #BANK 0
    Num_Ones = 256
    arr[refline][767:767+Num_Ones] = 1 #BL3
    arr[refline+1][767:767+Num_Ones] = 1 #BL2
    arr[refline+2][767:767+Num_Ones] = 1 #BL1
    arr[refline+3][767:767+Num_Ones] = 1 #BL0

    #BANK 1
    arr[refline][511:511+Num_Ones] = 1 #BL3
    arr[refline+1][511:511+Num_Ones] = 1 #BL2
    arr[refline+2][511:511+Num_Ones] = 1 #BL1
    arr[refline+3][511:511+Num_Ones] = 1 #BL0

    #BANK 2
    arr[refline][255:255+Num_Ones] = 1 #BL3
    arr[refline+1][255:255+Num_Ones] = 1 #BL2
    arr[refline+2][255:255+Num_Ones] = 1 #BL1
    arr[refline+3][255:255+Num_Ones] = 1 #BL0

    #BANK 3
    arr[refline][0:0+Num_Ones] = 1 #BL3
    arr[refline+1][0:0+Num_Ones] = 1 #BL2
    arr[refline+2][0:0+Num_Ones] = 1 #BL1
    arr[refline+3][0:0+Num_Ones] = 1 #BL0
arr = arr.flatten(order='F')
'''

In [12]:
#SRAM IMC DATA
def SRAM_IMC_DATA(NUM_IN,Num_Zeros_Ref):
    arr = np.zeros((320, 1024),dtype=np.uint8)
    #Keep all the Delay lines at 1
    arr[8:][:] = 1  #0:7 are dedicated to Inputs.
    #Change Reflines to be at zero  
    for refline in range(24,280,36):
        #BANK 0
        Num_Zeros = Num_Zeros_Ref
        arr[refline,768:768+Num_Zeros] = 0 #BL3
        arr[refline+1,768:768+Num_Zeros] = 0 #BL2
        arr[refline+2,768:768+Num_Zeros] = 0 #BL1
        arr[refline+3,768:768+Num_Zeros] = 0 #BL0

        #BANK 1
        arr[refline,512:512+Num_Zeros] = 0 #BL3
        arr[refline+1,512:512+Num_Zeros] = 0 #BL2
        arr[refline+2,512:512+Num_Zeros] = 0 #BL1
        arr[refline+3,512:512+Num_Zeros] = 0 #BL0

        #BANK 2
        arr[refline,256:256+Num_Zeros] = 0 #BL3
        arr[refline+1,256:256+Num_Zeros] = 0 #BL2
        arr[refline+2,256:256+Num_Zeros] = 0 #BL1
        arr[refline+3,256:256+Num_Zeros] = 0 #BL0

        #BANK 3
        arr[refline,0:0+Num_Zeros] = 0 #BL3
        arr[refline+1,0:0+Num_Zeros] = 0 #BL2
        arr[refline+2,0:0+Num_Zeros] = 0 #BL1
        arr[refline+3,0:0+Num_Zeros] = 0 #BL0

    #Now give NUM_IN worth Input lines on.
    #Bank 3
    arr[4:8,0:0+NUM_IN] = 1
    arr[0:4,NUM_IN:256] = 1

    #Bank 2
    arr[4:8,256:256+NUM_IN] = 1
    arr[0:4,256+NUM_IN:512] = 1

    #Bank 1
    arr[4:8,512:512+NUM_IN] = 1
    arr[0:4,512+NUM_IN:768] = 1

    #Bank 0
    arr[4:8,768:768+NUM_IN] = 1
    arr[0:4,768+NUM_IN:1024] = 1
    arr = arr.flatten(order='F')
    return arr

In [13]:
#Initialise SRAM with default data.
write_data = SRAM_IMC_DATA(128,256).tolist()
ret_data = Utils.Write_SRAM_Masked(client_socket,write_data,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 10
Received Data Size: 1
All Data Received
[128]


In [14]:
#Turn on Analog PS
PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

['+2.99815000E+00', '+2.72590000E-02']


In [15]:
signal_array = [Utils.BANK_EN_0,Utils.BANK_EN_1,Utils.BANK_EN_2,Utils.BANK_EN_3,Utils.CTRL_EN,Utils.ENTIME,Utils.InputEN_DAC]
value_array = [1,1,1,1,1,1,1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
print(ret_data)

[128]


In [16]:
signal_array = [Utils.DFF_RST]
value_array = [1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
print(ret_data)

[128]


In [17]:
TDC_DATA = []
signal_array = [Utils.SCN_SEL_2,Utils.SCN_SEL_1,Utils.SCN_SEL_0]
value_array = Utils.TDC_OUT_SC_LM["value"] #Scan chain id
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
print(ret_data)
for Bank_Sel in [0,1]:
    for cs in range(0,4):
        #select the BL
        ret_data = set_CS(cs)
        #Turn On IN_EN
        signal_array = [Utils.IN_EN,Utils.BANK_SEL,Utils.SCN_IN]
        value_array = [1,Bank_Sel,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Sample the CLK_A 1
        signal_array = [Utils.CLK_A]
        value_array = [1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Sample the CLK_A 0
        signal_array = [Utils.CLK_A]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Sample the IN_EN 0
        signal_array = [Utils.IN_EN]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Call Scan Out
        scan_id = Utils.TDC_OUT_SC_LM["id"]
        scan_len = Utils.TDC_OUT_SC_LM["len"]
        ret_data = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,0)
        TDC_DATA.append(ret_data)
        print(ret_data) 

[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[132, 0, 64, 26, 49, 132, 19, 73, 20, 145, 36, 18, 78, 34, 145, 196, 19, 78, 56, 0, 68, 17, 65, 20, 145, 196, 17, 79, 16, 33, 68, 147, 70, 60, 177, 196, 145, 72, 2, 209, 164, 16, 75, 28, 97, 196, 18, 73, 26, 81, 196, 16, 67, 60, 241, 36, 17, 73, 50, 169, 164, 144, 78, 12, 97, 36, 17, 71, 34, 49, 164, 17, 77, 34, 169, 36, 18, 78, 60, 201]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[132, 0, 64, 42, 49, 132, 17, 65, 36, 145, 36, 16, 78, 34, 17, 196, 19, 70, 56, 0, 68, 19, 65, 20, 17, 196, 18, 79, 16, 193, 68, 145, 74, 60, 49, 196, 18, 79, 60, 209, 36, 19, 77, 60, 33, 196, 16, 65, 10, 17, 68, 18, 67, 28, 241, 36, 17, 78, 18, 105, 36, 147, 78, 12, 161, 36, 16, 67, 60, 209, 164, 18, 77, 2, 169, 36, 18, 70, 28, 201]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Receiv

In [18]:
signal_array = [Utils.SCN_SEL_2,Utils.SCN_SEL_1,Utils.SCN_SEL_0]
value_array = Utils.TDC_OUT_SC_RM["value"] #Scan chain id
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)
for Bank_Sel in [0,1]:
    for cs in range(0,4):
        #select the BL
        ret_data = set_CS(cs)
        #Turn On IN_EN
        signal_array = [Utils.IN_EN,Utils.BANK_SEL,Utils.SCN_IN]
        value_array = [1,Bank_Sel,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Sample the CLK_A 1
        signal_array = [Utils.CLK_A]
        value_array = [1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Sample the CLK_A 0
        signal_array = [Utils.CLK_A]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Sample the IN_EN 0
        signal_array = [Utils.IN_EN]
        value_array = [0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #Call Scan Out
        scan_id = Utils.TDC_OUT_SC_RM["id"]
        scan_len = Utils.TDC_OUT_SC_RM["len"]
        ret_data = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,0)
        TDC_DATA.append(ret_data)
        print(ret_data) 

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[196, 18, 65, 10, 201, 68, 19, 71, 36, 41, 164, 145, 72, 60, 41, 196, 18, 74, 60, 177, 36, 2, 64, 44, 73, 36, 19, 79, 58, 241, 68, 19, 69, 10, 113, 36, 147, 68, 24, 241, 68, 16, 66, 34, 41, 164, 146, 68, 24, 241, 196, 144, 76, 28, 41, 196, 19, 79, 26, 233, 196, 18, 78, 52, 201, 196, 147, 8, 0, 97, 68, 146, 68, 58, 241, 36, 19, 73, 42, 177]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[196, 16, 65, 50, 73, 68, 17, 75, 4, 201, 164, 16, 79, 60, 201, 68, 19, 66, 28, 49, 36, 2, 64, 12, 137, 36, 19, 79, 26, 113, 68, 19, 65, 50, 177, 36, 147, 72, 40, 241, 132, 19, 66, 60, 201, 36, 147, 72, 40, 113, 68, 147, 68, 12, 201, 196, 17, 75, 42, 233, 196, 16, 70, 20, 201, 196, 145, 8, 0, 161, 68, 144, 72, 26, 241, 36, 19, 78

In [19]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Default Signals Set
[128]


In [20]:
def TDC_OUT_u8_to_dec(TDC_OUT):
  arr = np.array(TDC_OUT, dtype=np.uint8)
  binary_strings = np.vectorize(np.binary_repr)(arr, width=8)
  binary_strings_rev = np.vectorize(lambda x: x[::-1])(binary_strings)
  all_bits = ''.join(binary_strings_rev)

  groups_of_10 = [all_bits[i:i+10] for i in range(0, len(all_bits), 10)]

  def signed_10bit_to_int(bin_str):
    sign_bit = bin_str[0]
    magnitude = int(bin_str[1:], 2)
    return magnitude if sign_bit == '0' else -magnitude

  signed_integers = [signed_10bit_to_int(bits) for bits in groups_of_10]

  return signed_integers

In [21]:
TDC_DEC_DATA = []
for i in range(16):
    TDC_DEC_DATA.append(TDC_OUT_u8_to_dec(TDC_DATA[i]))

In [22]:
for i in range(16):
    print(TDC_DEC_DATA[i])

[132, 0, 150, 140, 135, 137, 138, 137, 145, 135, 145, 137, 143, 135, 135, 0, 138, 136, 138, 137, 142, 143, 130, 132, 139, 150, 143, 141, 142, 145, 144, 139, 148, 141, 142, 134, 141, 137, 150, 138, 140, 140, 143, 143, 146, 137, 147, 149, 148, 151, 140, 134, 146, 142, 145, 140, 150, 139, 145, 149, 145, 135, 143, 147]
[132, 0, 149, 140, 134, 136, 137, 137, 144, 135, 145, 136, 143, 134, 135, 0, 139, 136, 138, 136, 141, 143, 130, 131, 138, 149, 143, 140, 141, 143, 143, 139, 147, 139, 143, 132, 140, 136, 148, 136, 137, 140, 142, 143, 146, 135, 146, 150, 147, 151, 140, 133, 144, 140, 143, 139, 149, 139, 144, 149, 145, 134, 142, 147]
[132, 0, 149, 139, 133, 135, 137, 137, 143, 134, 144, 135, 143, 134, 135, 0, 138, 135, 137, 137, 141, 142, 130, 131, 137, 148, 142, 139, 140, 143, 143, 139, 147, 139, 142, 132, 140, 136, 148, 136, 138, 139, 141, 142, 145, 136, 146, 148, 147, 149, 138, 133, 144, 140, 143, 139, 149, 138, 143, 148, 144, 134, 141, 146]
[131, 0, 148, 138, 133, 135, 136, 136, 143, 134, 

In [ ]:
import matplotlib.pyplot as plt
for i in range(16):
    plt.plot(TDC_DEC_DATA[i])

In [ ]:
arr_write = write_data
u8_array = []
for i in range(0, len(arr_write), 8):
    byte_bits = arr_write[i:i+8]
    value = sum(bit << j for j, bit in enumerate(byte_bits))  # index 0 is LSB
    u8_array.append(value)

print(u8_array)

In [ ]:
#Call Read SRAM here
SRAM_DATA = Utils.Read_SRAM(client_socket,1)
print(len(SRAM_DATA))
print(SRAM_DATA)

In [ ]:
if SRAM_DATA == u8_array:
    print("Lists are equal")
else:
    print("Lists are not equal")

In [ ]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)


In [23]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)

0